In [2]:

'''# RouteRanker — Module 2: Feature Engineering

## Project Overview

RouteRanker is a public transportation data analysis project that uses GTFS (General Transit Feed Specification) data to study transit routes, scheduled trips, stops, operating days, and route characteristics across six Indian cities.

## Objectives

- Inspect the available GTFS datasets.
- Identify missing values and data quality issues.
- Extract time-based features from scheduled transit data.
- Calculate route profiles and hourly departure counts.
- Analyse operating days and estimate route distances.
- Combine the engineered features into a final dataset.
- Document the features, data quality checks, assumptions, and limitations.

## Tools and Technologies

- Python
- Pandas and NumPy
- Jupyter Notebook
- GTFS datasets
- VS Code'''


'# RouteRanker — Module 2: Feature Engineering\n\n## Project Overview\n\nRouteRanker is a public transportation data analysis project that uses GTFS (General Transit Feed Specification) data to study transit routes, scheduled trips, stops, operating days, and route characteristics across six Indian cities.\n\n## Objectives\n\n- Inspect the available GTFS datasets.\n- Identify missing values and data quality issues.\n- Extract time-based features from scheduled transit data.\n- Calculate route profiles and hourly departure counts.\n- Analyse operating days and estimate route distances.\n- Combine the engineered features into a final dataset.\n- Document the features, data quality checks, assumptions, and limitations.\n\n## Tools and Technologies\n\n- Python\n- Pandas and NumPy\n- Jupyter Notebook\n- GTFS datasets\n- VS Code'

In [ ]:
## 1. Check GTFS File Availability Across Cities

from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")

required_files = [
    "routes.txt",
    "trips.txt",
    "stops.txt",
    "stop_times.txt",
    "calendar.txt",
    "calendar_dates.txt",
    "shapes.txt"
]

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    print("\n---", city_folder.name, "---")

    for file_name in required_files:
        file_path = city_folder / file_name

        if file_path.exists():
            print(file_name, ": FOUND")
        else:
            print(file_name, ": NOT FOUND")



--- Ahmedabad ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : NOT FOUND
shapes.txt : FOUND

--- Chennai ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : FOUND
shapes.txt : FOUND

--- Hyderabad ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : NOT FOUND
shapes.txt : NOT FOUND

--- Mumbai ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : NOT FOUND
shapes.txt : NOT FOUND

--- Pune ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : NOT FOUND
shapes.txt : FOUND

--- Rajkot ---
routes.txt : FOUND
trips.txt : FOUND
stops.txt : FOUND
stop_times.txt : FOUND
calendar.txt : FOUND
calendar_dates.txt : NOT FOUND
shapes.txt : NOT FOU

In [ ]:
## 2. Summarize Available GTFS Files
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")

required_files = [
    "routes.txt",
    "trips.txt",
    "stops.txt",
    "stop_times.txt",
    "calendar.txt",
    "calendar_dates.txt",
    "shapes.txt"
]

summary = []

for city_folder in sorted(raw_folder.iterdir()):
    if city_folder.is_dir():
        row = {"City": city_folder.name}

        for file_name in required_files:
            row[file_name] = (
                "FOUND"
                if (city_folder / file_name).exists()
                else "MISSING"
            )

        summary.append(row)

file_summary = pd.DataFrame(summary)

display(file_summary)


,City,routes.txt,trips.txt,stops.txt,stop_times.txt,calendar.txt,calendar_dates.txt,shapes.txt
0,Ahmedabad,FOUND,FOUND,FOUND,FOUND,FOUND,MISSING,FOUND
1,Chennai,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
2,Hyderabad,FOUND,FOUND,FOUND,FOUND,FOUND,MISSING,MISSING
3,Mumbai,FOUND,FOUND,FOUND,FOUND,FOUND,MISSING,MISSING
4,Pune,FOUND,FOUND,FOUND,FOUND,FOUND,MISSING,FOUND
5,Rajkot,FOUND,FOUND,FOUND,FOUND,FOUND,MISSING,MISSING


In [ ]:
## 3. Analyze Initial Data Quality and Missing Values
from pathlib import Path
import pandas as pd
import gc

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

gtfs_files = [
    "routes.txt",
    "trips.txt",
    "stops.txt",
    "stop_times.txt",
    "calendar.txt",
    "calendar_dates.txt",
    "shapes.txt"
]

quality_rows = []
missing_rows = []

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    for file_name in gtfs_files:
        file_path = city_folder / file_name

        if not file_path.exists():
            continue

        print("Checking:", city_folder.name, file_name)

        df = pd.read_csv(file_path, low_memory=False)

        missing_counts = df.isnull().sum()

        quality_rows.append({
            "City": city_folder.name,
            "File": file_name,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Duplicate_Rows": int(df.duplicated().sum()),
            "Total_Missing_Cells": int(missing_counts.sum())
        })

        for column, count in missing_counts.items():
            if count > 0:
                missing_rows.append({
                    "City": city_folder.name,
                    "File": file_name,
                    "Column": column,
                    "Missing_Count": int(count),
                    "Missing_Percent": round(
                        count / len(df) * 100, 2
                    ) if len(df) else 0
                })

        del df
        gc.collect()

quality_df = pd.DataFrame(quality_rows)
missing_df = pd.DataFrame(missing_rows)

quality_df.to_csv(
    output_folder / "module2_data_quality_report.csv",
    index=False
)

missing_df.to_csv(
    output_folder / "module2_missing_values.csv",
    index=False
)

print("\nDATA QUALITY SUMMARY")
display(quality_df)

print("\nMISSING VALUES SUMMARY")
display(missing_df.head(30))

print("\nReports saved in:", output_folder)


Checking: Ahmedabad routes.txt
Checking: Ahmedabad trips.txt
Checking: Ahmedabad stops.txt
Checking: Ahmedabad stop_times.txt
Checking: Ahmedabad calendar.txt
Checking: Ahmedabad shapes.txt
Checking: Chennai routes.txt
Checking: Chennai trips.txt
Checking: Chennai stops.txt
Checking: Chennai stop_times.txt
Checking: Chennai calendar.txt
Checking: Chennai calendar_dates.txt
Checking: Chennai shapes.txt
Checking: Hyderabad routes.txt
Checking: Hyderabad trips.txt
Checking: Hyderabad stops.txt
Checking: Hyderabad stop_times.txt
Checking: Hyderabad calendar.txt
Checking: Mumbai routes.txt
Checking: Mumbai trips.txt
Checking: Mumbai stops.txt
Checking: Mumbai stop_times.txt
Checking: Mumbai calendar.txt
Checking: Pune routes.txt
Checking: Pune trips.txt
Checking: Pune stops.txt
Checking: Pune stop_times.txt
Checking: Pune calendar.txt
Checking: Pune shapes.txt
Checking: Rajkot routes.txt
Checking: Rajkot trips.txt
Checking: Rajkot stops.txt
Checking: Rajkot stop_times.txt
Checking: Rajkot c

,City,File,Rows,Columns,Duplicate_Rows,Total_Missing_Cells
0,Ahmedabad,routes.txt,639,5,0,0
1,Ahmedabad,trips.txt,14323,6,0,14323
2,Ahmedabad,stops.txt,3203,4,0,0
3,Ahmedabad,stop_times.txt,646471,6,0,0
4,Ahmedabad,calendar.txt,1,10,0,0
5,Ahmedabad,shapes.txt,270043,4,0,0
6,Chennai,routes.txt,5402,8,0,15421
7,Chennai,trips.txt,48435,6,0,96245
8,Chennai,stops.txt,6542,10,0,35661
9,Chennai,stop_times.txt,1389081,5,0,0



MISSING VALUES SUMMARY


,City,File,Column,Missing_Count,Missing_Percent
0,Ahmedabad,trips.txt,shape_id,14323,100.00
1,Chennai,routes.txt,agency_id,28,0.52
2,Chennai,routes.txt,route_desc,4639,85.88
3,Chennai,routes.txt,route_color,5380,99.59
4,Chennai,routes.txt,route_text_color,5374,99.48
5,Chennai,trips.txt,direction_id,1388,2.87
6,Chennai,trips.txt,trip_headsign,47047,97.13
7,Chennai,trips.txt,shape_id,47810,98.71
8,Chennai,stops.txt,stop_desc,6330,96.76
9,Chennai,stops.txt,zone_id,6288,96.12



Reports saved in: data\processed


In [ ]:
## 4. Create Time-Based Features from Stop Times
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)


def parse_gtfs_time(series):
    parts = series.astype("string").str.strip().str.extract(
        r"^(\d+):(\d{2}):(\d{2})$"
    )

    hour = pd.to_numeric(parts[0], errors="coerce")
    minute = pd.to_numeric(parts[1], errors="coerce")
    second = pd.to_numeric(parts[2], errors="coerce")

    valid = (
        hour.notna()
        & minute.between(0, 59)
        & second.between(0, 59)
    )

    hour = hour.where(valid)
    minute = minute.where(valid)

    return pd.DataFrame({
        "gtfs_hour": hour,
        "minute": minute,
        "clock_hour": hour % 24,
        "total_minutes": hour * 60 + minute
    }, index=series.index)


def get_time_period(hours):
    period = pd.Series(
        pd.NA, index=hours.index, dtype="string"
    )

    period.loc[hours.between(5, 11)] = "Morning"
    period.loc[hours.between(12, 16)] = "Afternoon"
    period.loc[hours.between(17, 20)] = "Evening"

    period.loc[
        hours.notna() & ~hours.between(5, 20)
    ] = "Night"

    return period


for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    file_path = city_folder / "stop_times.txt"

    if not file_path.exists():
        continue

    print("Processing:", city_folder.name)

    df = pd.read_csv(file_path, low_memory=False)

    for column in ["arrival_time", "departure_time"]:
        if column not in df.columns:
            continue

        parsed = parse_gtfs_time(df[column])

        df[f"{column}_hour"] = parsed["gtfs_hour"]
        df[f"{column}_minute"] = parsed["minute"]
        df[f"{column}_clock_hour"] = parsed["clock_hour"]
        df[f"{column}_total_minutes"] = parsed["total_minutes"]
        df[f"{column}_period"] = get_time_period(
            parsed["clock_hour"]
        )

    output_file = (
        output_folder
        / f"{city_folder.name.lower()}_stop_times_time_features.csv"
    )

    df.to_csv(output_file, index=False)

    print("Saved:", output_file)
    print("Rows:", len(df))

print("\nTime feature engineering completed.")


Processing: Ahmedabad
Saved: data\processed\ahmedabad_stop_times_time_features.csv
Rows: 646471
Processing: Chennai
Saved: data\processed\chennai_stop_times_time_features.csv
Rows: 1389081
Processing: Hyderabad
Saved: data\processed\hyderabad_stop_times_time_features.csv
Rows: 1132807
Processing: Mumbai
Saved: data\processed\mumbai_stop_times_time_features.csv
Rows: 2655026
Processing: Pune
Saved: data\processed\pune_stop_times_time_features.csv
Rows: 429302
Processing: Rajkot
Saved: data\processed\rajkot_stop_times_time_features.csv
Rows: 94809

Time feature engineering completed.


In [ ]:
## 5. Generate Basic Route Profiles for All Cities
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

all_profiles = []

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    print("Processing:", city_folder.name)

    routes_path = city_folder / "routes.txt"
    trips_path = city_folder / "trips.txt"
    stop_times_path = city_folder / "stop_times.txt"
    stops_path = city_folder / "stops.txt"

    # Read only the columns needed for this calculation
    routes = pd.read_csv(
        routes_path,
        dtype={"route_id": "string"},
        low_memory=False
    )

    trips = pd.read_csv(
        trips_path,
        usecols=["trip_id", "route_id"],
        dtype={"trip_id": "string", "route_id": "string"},
        low_memory=False
    )

    stop_times = pd.read_csv(
        stop_times_path,
        usecols=["trip_id", "stop_id"],
        dtype={"trip_id": "string", "stop_id": "string"},
        low_memory=False
    )

    stops = pd.read_csv(
        stops_path,
        usecols=["stop_id"],
        dtype={"stop_id": "string"},
        low_memory=False
    )

    # Keep one record per route ID and trip ID
    routes = routes.dropna(subset=["route_id"])
    routes = routes.drop_duplicates(
        subset=["route_id"], keep="first"
    )

    trips = trips.dropna(subset=["trip_id", "route_id"])
    trips = trips.drop_duplicates()

    # Check if any trip ID is linked to multiple routes
    conflicting = (
        trips.groupby("trip_id")["route_id"].nunique() > 1
    ).sum()

    if conflicting:
        print("Trip IDs linked to multiple routes:", conflicting)

    trips = trips.drop_duplicates(
        subset=["trip_id"], keep="first"
    )

    # Count distinct scheduled trips per route
    trip_counts = (
        trips.groupby("route_id")["trip_id"]
        .nunique()
        .reset_index(name="distinct_trip_count")
    )

    # Connect stop-time records to their routes
    stop_links = stop_times.dropna(
        subset=["trip_id", "stop_id"]
    ).merge(
        trips[["trip_id", "route_id"]],
        on="trip_id",
        how="inner"
    )

    # Count unique stop IDs referenced by each route
    stop_counts = (
        stop_links.groupby("route_id")["stop_id"]
        .nunique()
        .reset_index(name="distinct_stop_ids_referenced")
    )

    # Count only stop IDs that exist in stops.txt
    known_stop_ids = set(stops["stop_id"].dropna())

    known_links = stop_links[
        stop_links["stop_id"].isin(known_stop_ids)
    ]

    known_counts = (
        known_links.groupby("route_id")["stop_id"]
        .nunique()
        .reset_index(name="distinct_known_stops_served")
    )

    # Preserve route information where available
    for column in [
        "route_short_name", "route_long_name", "route_type"
    ]:
        if column not in routes.columns:
            routes[column] = pd.NA

    short_name = (
        routes["route_short_name"].astype("string")
        .fillna("").str.strip()
    )
    long_name = (
        routes["route_long_name"].astype("string")
        .fillna("").str.strip()
    )

    routes["route_name"] = short_name.where(
        short_name.ne(""), long_name
    ).replace("", pd.NA)

    profile = routes[
        [
            "route_id", "route_name", "route_short_name",
            "route_long_name", "route_type"
        ]
    ].copy()

    profile = profile.merge(
        trip_counts, on="route_id", how="left"
    )
    profile = profile.merge(
        stop_counts, on="route_id", how="left"
    )
    profile = profile.merge(
        known_counts, on="route_id", how="left"
    )

    count_columns = [
        "distinct_trip_count",
        "distinct_stop_ids_referenced",
        "distinct_known_stops_served"
    ]

    profile[count_columns] = (
        profile[count_columns].fillna(0).astype("int64")
    )

    profile.insert(0, "city", city_folder.name)

    all_profiles.append(profile)

# Combine all six cities into one dataset
final_profiles = pd.concat(
    all_profiles, ignore_index=True
)

output_file = (
    output_folder / "module2_route_profiles_basic_all_cities.csv"
)

final_profiles.to_csv(output_file, index=False)

print("\nRoute profile calculation completed.")
print("Total route records:", len(final_profiles))
print("Output saved to:", output_file)

display(final_profiles.head(10))


Processing: Ahmedabad
Processing: Chennai
Processing: Hyderabad
Processing: Mumbai
Processing: Pune
Processing: Rajkot

Route profile calculation completed.
Total route records: 8472
Output saved to: data\processed\module2_route_profiles_basic_all_cities.csv


,city,route_id,route_name,route_short_name,route_long_name,route_type,distinct_trip_count,distinct_stop_ids_referenced,distinct_known_stops_served
0,Ahmedabad,101sh,101sh,101sh,P.D.P.U. Cross Road ⇆ Asarva Railway Station,3,10,44,44
1,Ahmedabad,103-1,103,103,Bhakti Circle ⇆ Gujarat University,3,22,80,80
2,Ahmedabad,103-2,103,103,Bhakti Circle ⇆ Sarangpur,3,14,52,52
3,Ahmedabad,103-3,103,103,Gujarat University ⇆ Sarangpur,3,6,28,28
4,Ahmedabad,105,105,105,Naroda Industrial Township( MuthiyaGam ) ⇆ Lal...,3,216,40,40
5,Ahmedabad,107,107,107,Sarangpur ⇆ Sarangpur,3,6,91,91
6,Ahmedabad,109-1,109,109,Amba Township Sector-5 (Trimandir) ⇆ G.E.B (Na...,3,6,79,79
7,Ahmedabad,109-2,109,109,Amba Township Sector-5 (Trimandir) ⇆ Naroda,3,3,63,63
8,Ahmedabad,109-3,109,109,Chandkheda Gam ⇆ Amba Township Sector-5 (Trima...,3,1,24,24
9,Ahmedabad,109-4,109,109,Chandkheda Gam ⇆ Naroda,3,4,40,40


In [ ]:
## 6. Analyze Scheduled Departures by Hour
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

hourly_results = []

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    print("Processing:", city_folder.name)

    trips_path = city_folder / "trips.txt"
    stop_times_path = city_folder / "stop_times.txt"

    trips = pd.read_csv(
        trips_path,
        usecols=["trip_id", "route_id"],
        dtype={"trip_id": "string", "route_id": "string"},
        low_memory=False
    )

    stop_times = pd.read_csv(
        stop_times_path,
        usecols=["trip_id", "stop_sequence", "departure_time"],
        dtype={"trip_id": "string"},
        low_memory=False
    )

    trips = trips.dropna(subset=["trip_id", "route_id"])
    trips = trips.drop_duplicates(subset=["trip_id"])

    # Convert stop sequence to a number
    stop_times["stop_sequence"] = pd.to_numeric(
        stop_times["stop_sequence"], errors="coerce"
    )

    # Select the first stop of each trip
    first_stops = (
        stop_times.dropna(subset=["trip_id", "stop_sequence"])
        .sort_values(["trip_id", "stop_sequence"])
        .drop_duplicates(subset=["trip_id"], keep="first")
        .copy()
    )

    # Attach route IDs to the first stop of each trip
    first_stops = first_stops.merge(
        trips,
        on="trip_id",
        how="inner"
    )

    # Parse GTFS times, including times beyond 24:00
    parts = first_stops["departure_time"].astype(
        "string"
    ).str.strip().str.extract(
        r"^(\d+):([0-5]\d):([0-5]\d)$"
    )

    gtfs_hour = pd.to_numeric(parts[0], errors="coerce")
    minute = pd.to_numeric(parts[1], errors="coerce")

    valid = gtfs_hour.notna() & minute.notna()

    first_stops["departure_clock_hour"] = (
        gtfs_hour % 24
    ).where(valid)

    # Count each trip once by route and clock hour
    valid_departures = first_stops.dropna(
        subset=["departure_clock_hour"]
    ).copy()

    valid_departures["departure_clock_hour"] = (
        valid_departures["departure_clock_hour"].astype(int)
    )

    hourly = (
        valid_departures.groupby(
            ["route_id", "departure_clock_hour"]
        )["trip_id"]
        .nunique()
        .reset_index(name="scheduled_trip_count")
    )

    hourly.insert(0, "city", city_folder.name)
    hourly_results.append(hourly)

# Combine results from all cities
hourly_df = pd.concat(hourly_results, ignore_index=True)

output_file = (
    output_folder / "module2_hourly_trip_counts_all_cities.csv"
)

hourly_df.to_csv(output_file, index=False)

print("\nHourly trip analysis completed.")
print("Total route-hour records:", len(hourly_df))
print("Saved to:", output_file)

display(hourly_df.head(15))


Processing: Ahmedabad
Processing: Chennai
Processing: Hyderabad
Processing: Mumbai
Processing: Pune
Processing: Rajkot

Hourly trip analysis completed.
Total route-hour records: 51036
Saved to: data\processed\module2_hourly_trip_counts_all_cities.csv


,city,route_id,departure_clock_hour,scheduled_trip_count
0,Ahmedabad,101-C,6,7
1,Ahmedabad,101-C,7,10
2,Ahmedabad,101-C,8,9
3,Ahmedabad,101-C,9,7
4,Ahmedabad,101-C,10,6
5,Ahmedabad,101-C,11,6
6,Ahmedabad,101-C,12,3
7,Ahmedabad,101-C,13,4
8,Ahmedabad,101-C,14,3
9,Ahmedabad,101-C,15,4


In [ ]:
## 7. Estimate Route Distances
from pathlib import Path
import pandas as pd
import numpy as np

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)


def haversine_km(lat1, lon1, lat2, lon2):
    # Approximate distance between two geographic coordinates
    earth_radius = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0, 1)

    return 2 * earth_radius * np.arcsin(np.sqrt(a))


distance_results = []

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    print("Processing:", city_folder.name)

    trips_path = city_folder / "trips.txt"
    stop_times_path = city_folder / "stop_times.txt"
    stops_path = city_folder / "stops.txt"
    shapes_path = city_folder / "shapes.txt"

    trips = pd.read_csv(trips_path, low_memory=False)

    needed_trip_columns = ["trip_id", "route_id"]

    if "shape_id" in trips.columns:
        needed_trip_columns.append("shape_id")

    trips = trips[needed_trip_columns].copy()
    trips = trips.dropna(subset=["trip_id", "route_id"])
    trips = trips.drop_duplicates(subset=["trip_id"])

    route_shape_distances = pd.DataFrame(
        columns=["route_id", "estimated_route_length_km"]
    )

    # First choice: use usable shape geometry
    if shapes_path.exists() and "shape_id" in trips.columns:
        shapes = pd.read_csv(shapes_path, low_memory=False)

        required_shape_columns = [
            "shape_id", "shape_pt_lat",
            "shape_pt_lon", "shape_pt_sequence"
        ]

        if all(c in shapes.columns for c in required_shape_columns):
            shapes = shapes[required_shape_columns].copy()

            for col in [
                "shape_pt_lat", "shape_pt_lon",
                "shape_pt_sequence"
            ]:
                shapes[col] = pd.to_numeric(
                    shapes[col], errors="coerce"
                )

            shapes = shapes.dropna(
                subset=[
                    "shape_id", "shape_pt_lat",
                    "shape_pt_lon", "shape_pt_sequence"
                ]
            )

            shapes = shapes[
                shapes["shape_pt_lat"].between(-90, 90)
                & shapes["shape_pt_lon"].between(-180, 180)
            ]

            shapes = shapes.sort_values(
                ["shape_id", "shape_pt_sequence"]
            )

            # Get the previous point along each shape
            shapes["previous_lat"] = (
                shapes.groupby("shape_id")["shape_pt_lat"].shift()
            )
            shapes["previous_lon"] = (
                shapes.groupby("shape_id")["shape_pt_lon"].shift()
            )

            valid_segments = shapes.dropna(
                subset=["previous_lat", "previous_lon"]
            ).copy()

            if not valid_segments.empty:
                valid_segments["segment_km"] = haversine_km(
                    valid_segments["previous_lat"],
                    valid_segments["previous_lon"],
                    valid_segments["shape_pt_lat"],
                    valid_segments["shape_pt_lon"]
                )

                shape_lengths = (
                    valid_segments.groupby("shape_id")
                    .agg(
                        shape_length_km=("segment_km", "sum"),
                        valid_segments=("segment_km", "count")
                    )
                    .reset_index()
                )

                shape_lengths = shape_lengths[
                    (shape_lengths["valid_segments"] > 0)
                    & (shape_lengths["shape_length_km"] > 0)
                ]

                trip_shapes = trips.dropna(
                    subset=["shape_id"]
                )[["route_id", "shape_id"]].drop_duplicates()

                route_shape_distances = (
                    trip_shapes.merge(
                        shape_lengths,
                        on="shape_id",
                        how="inner"
                    )
                    .groupby("route_id")["shape_length_km"]
                    .mean()
                    .reset_index(
                        name="estimated_route_length_km"
                    )
                )

    route_shape_distances["distance_method"] = "GTFS shape geometry"

    # Fallback: estimate distance from consecutive stop coordinates
    stop_distances = pd.DataFrame(
        columns=["route_id", "fallback_distance_km"]
    )

    if (
        "stop_id" in pd.read_csv(
            stop_times_path, nrows=0
        ).columns
        and "stop_sequence" in pd.read_csv(
            stop_times_path, nrows=0
        ).columns
    ):
        stop_times = pd.read_csv(
            stop_times_path,
            usecols=["trip_id", "stop_id", "stop_sequence"],
            low_memory=False
        )

        stops = pd.read_csv(
            stops_path,
            usecols=["stop_id", "stop_lat", "stop_lon"],
            low_memory=False
        )

        stops["stop_lat"] = pd.to_numeric(
            stops["stop_lat"], errors="coerce"
        )
        stops["stop_lon"] = pd.to_numeric(
            stops["stop_lon"], errors="coerce"
        )

        stops = stops[
            stops["stop_lat"].between(-90, 90)
            & stops["stop_lon"].between(-180, 180)
        ].dropna(subset=["stop_id"])

        stop_times["stop_sequence"] = pd.to_numeric(
            stop_times["stop_sequence"], errors="coerce"
        )

        stop_times = stop_times.dropna(
            subset=["trip_id", "stop_id", "stop_sequence"]
        )

        trip_stops = stop_times.merge(
            trips[["trip_id", "route_id"]],
            on="trip_id",
            how="inner"
        ).merge(
            stops,
            on="stop_id",
            how="inner"
        )

        trip_stops = trip_stops.sort_values(
            ["trip_id", "stop_sequence"]
        )

        trip_stops["previous_lat"] = (
            trip_stops.groupby("trip_id")["stop_lat"].shift()
        )
        trip_stops["previous_lon"] = (
            trip_stops.groupby("trip_id")["stop_lon"].shift()
        )

        segments = trip_stops.dropna(
            subset=["previous_lat", "previous_lon"]
        ).copy()

        if not segments.empty:
            segments["segment_km"] = haversine_km(
                segments["previous_lat"],
                segments["previous_lon"],
                segments["stop_lat"],
                segments["stop_lon"]
            )

            trip_lengths = (
                segments.groupby(["route_id", "trip_id"])
                .agg(
                    trip_length_km=("segment_km", "sum"),
                    segment_count=("segment_km", "count")
                )
                .reset_index()
            )

            trip_lengths = trip_lengths[
                (trip_lengths["segment_count"] > 0)
                & (trip_lengths["trip_length_km"] > 0)
            ]

            stop_distances = (
                trip_lengths.groupby("route_id")["trip_length_km"]
                .median()
                .reset_index(name="fallback_distance_km")
            )

    # Prefer shape-based distance; use stop coordinates only
    # for routes without a usable shape estimate.
    result = stop_distances.merge(
        route_shape_distances[
            ["route_id", "estimated_route_length_km",
             "distance_method"]
        ],
        on="route_id",
        how="outer"
    )

    if "estimated_route_length_km" not in result.columns:
        result["estimated_route_length_km"] = np.nan

    if "distance_method" not in result.columns:
        result["distance_method"] = pd.NA

    use_fallback = result["estimated_route_length_km"].isna()

    result.loc[use_fallback, "estimated_route_length_km"] = (
        result.loc[use_fallback, "fallback_distance_km"]
    )

    result.loc[use_fallback, "distance_method"] = (
        "Straight-line stop-coordinate estimate"
    )

    result = result.drop(columns=["fallback_distance_km"])

    result.insert(0, "city", city_folder.name)
    distance_results.append(result)

distance_df = pd.concat(
    distance_results, ignore_index=True
)

distance_df["estimated_route_length_km"] = (
    distance_df["estimated_route_length_km"].round(3)
)

output_file = (
    output_folder / "module2_route_distance_estimates.csv"
)

distance_df.to_csv(output_file, index=False)

print("\nRoute distance estimation completed.")
print("Routes with estimates:",
      distance_df["estimated_route_length_km"].notna().sum())
print("Saved to:", output_file)

display(distance_df.head(10))


Processing: Ahmedabad
Processing: Chennai
Processing: Hyderabad
Processing: Mumbai
Processing: Pune
Processing: Rajkot

Route distance estimation completed.
Routes with estimates: 7461
Saved to: data\processed\module2_route_distance_estimates.csv


,city,route_id,estimated_route_length_km,distance_method
0,Ahmedabad,101-C,25.562884,Straight-line stop-coordinate estimate
1,Ahmedabad,101-C-PR,2.639218,Straight-line stop-coordinate estimate
2,Ahmedabad,101-C-RS,3.052088,Straight-line stop-coordinate estimate
3,Ahmedabad,101S-RM,9.443957,Straight-line stop-coordinate estimate
4,Ahmedabad,101U-NR,13.226289,Straight-line stop-coordinate estimate
5,Ahmedabad,101sh,44.528508,Straight-line stop-coordinate estimate
6,Ahmedabad,102-1,38.701152,Straight-line stop-coordinate estimate
7,Ahmedabad,102-2,43.644863,Straight-line stop-coordinate estimate
8,Ahmedabad,102-3,49.669578,Straight-line stop-coordinate estimate
9,Ahmedabad,103-1,22.365567,Straight-line stop-coordinate estimate


In [ ]:
## 8. Analyze Scheduled Operating Days
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

days = [
    "monday", "tuesday", "wednesday",
    "thursday", "friday", "saturday", "sunday"
]

all_city_results = []

for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    calendar_path = city_folder / "calendar.txt"
    trips_path = city_folder / "trips.txt"

    if not calendar_path.exists() or not trips_path.exists():
        print("Skipping:", city_folder.name,
              "- calendar.txt or trips.txt missing")
        continue

    print("Processing:", city_folder.name)

    trips = pd.read_csv(
        trips_path,
        usecols=["route_id", "service_id"],
        dtype={"route_id": "string", "service_id": "string"},
        low_memory=False
    )

    calendar = pd.read_csv(
        calendar_path,
        dtype={"service_id": "string"},
        low_memory=False
    )

    if not all(
        col in calendar.columns
        for col in ["service_id"] + days
    ):
        print("Skipping:", city_folder.name,
              "- required calendar columns missing")
        continue

    # Keep distinct route and service combinations
    route_services = trips[
        ["route_id", "service_id"]
    ].dropna(subset=["route_id"]).drop_duplicates()

    # Count distinct service IDs used by each route
    service_counts = (
        route_services.dropna(subset=["service_id"])
        .groupby("route_id")["service_id"]
        .nunique()
        .reset_index(name="service_ids_used")
    )

    # Count trips with a missing service ID
    missing_service = (
        trips.assign(
            missing_service=trips["service_id"].isna()
        )
        .groupby("route_id")["missing_service"]
        .sum()
        .reset_index(name="trips_missing_service_id")
    )

    # Prepare calendar weekday flags
    for day in days:
        calendar[day] = pd.to_numeric(
            calendar[day], errors="coerce"
        ).where(lambda x: x.isin([0, 1]))

    calendar = calendar.drop_duplicates(
        subset=["service_id"], keep="first"
    )

    # Match route services to calendar records
    matched = route_services.dropna(
        subset=["service_id"]
    ).merge(
        calendar[["service_id"] + days],
        on="service_id",
        how="left",
        indicator=True
    )

    matched["calendar_found"] = (
        matched["_merge"] == "both"
    )

    coverage = (
        matched.groupby("route_id")
        .agg(
            service_ids_found_in_calendar=(
                "calendar_found", "sum"
            )
        )
        .reset_index()
    )

    # Summarize days served by each route
    known = matched[matched["calendar_found"]].copy()

    if not known.empty:
        day_summary = (
            known.groupby("route_id")[days]
            .max()
            .reset_index()
        )

        # Only report day patterns when every used service
        # has a calendar record and all seven flags are valid.
        known_day_counts = (
            known.groupby("route_id")[days]
            .count()
            .min(axis=1)
            .rename("valid_day_flags")
            .reset_index()
        )

        day_summary = day_summary.merge(
            known_day_counts, on="route_id", how="left"
        )
    else:
        day_summary = pd.DataFrame(columns=["route_id"] + days)

    result = service_counts.merge(
        coverage, on="route_id", how="left"
    )

    result = result.merge(
        day_summary, on="route_id", how="left"
    )

    result = result.merge(
        missing_service, on="route_id", how="left"
    )

    result["service_ids_found_in_calendar"] = (
        result["service_ids_found_in_calendar"].fillna(0)
    )

    result["calendar_coverage_percent"] = (
        result["service_ids_found_in_calendar"]
        / result["service_ids_used"].replace(0, float("nan"))
        * 100
    ).round(2)

    complete_coverage = (
        result["service_ids_used"].notna()
        & result["service_ids_used"].gt(0)
        & result["service_ids_found_in_calendar"].eq(
            result["service_ids_used"]
        )
        & result.get(
            "valid_day_flags", pd.Series(index=result.index, dtype="float64")
        ).eq(7)
    )

    # Avoid claiming a complete weekly schedule when
    # service information is missing or incomplete.
    for day in days:
        result.loc[~complete_coverage, day] = pd.NA

    result["scheduled_operating_days_per_week"] = (
        result[days].sum(axis=1, min_count=7)
    )

    result["weekday_service"] = result[
        ["monday", "tuesday", "wednesday", "thursday", "friday"]
    ].max(axis=1)

    result["weekend_service"] = result[
        ["saturday", "sunday"]
    ].max(axis=1)

    result.insert(0, "city", city_folder.name)

    all_city_results.append(result)

# Combine all cities
service_df = pd.concat(
    all_city_results, ignore_index=True
)

output_file = (
    output_folder / "module2_route_service_days_all_cities.csv"
)

service_df.to_csv(output_file, index=False)

print("\nOperating-day analysis completed.")
print("Total route records:", len(service_df))
print("Saved to:", output_file)

display(service_df.head(10))


Processing: Ahmedabad
Processing: Chennai
Processing: Hyderabad
Processing: Mumbai
Processing: Pune
Processing: Rajkot

Operating-day analysis completed.
Total route records: 7466
Saved to: data\processed\module2_route_service_days_all_cities.csv


,city,route_id,service_ids_used,service_ids_found_in_calendar,monday,tuesday,wednesday,thursday,friday,saturday,sunday,valid_day_flags,trips_missing_service_id,calendar_coverage_percent,scheduled_operating_days_per_week,weekday_service,weekend_service
0,Ahmedabad,101-C,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
1,Ahmedabad,101-C-PR,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
2,Ahmedabad,101-C-RS,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
3,Ahmedabad,101S-RM,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
4,Ahmedabad,101U-NR,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
5,Ahmedabad,101sh,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
6,Ahmedabad,102-1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
7,Ahmedabad,102-2,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
8,Ahmedabad,102-3,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN
9,Ahmedabad,103-1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,100.0,NaN,NaN,NaN


In [ ]:
## 9. Combine Features into the Final Dataset
from pathlib import Path
import pandas as pd
import numpy as np

processed = Path("data/processed")

# Load the files created in previous steps
profiles = pd.read_csv(
    processed / "module2_route_profiles_basic_all_cities.csv",
    dtype={"route_id": str}
)

service_days = pd.read_csv(
    processed / "module2_route_service_days_all_cities.csv",
    dtype={"route_id": str}
)

hourly = pd.read_csv(
    processed / "module2_hourly_trip_counts_all_cities.csv",
    dtype={"route_id": str}
)

distances = pd.read_csv(
    processed / "module2_route_distance_estimates.csv",
    dtype={"route_id": str}
)

# Standardize city and route identifiers
for df in [profiles, service_days, hourly, distances]:
    df["city"] = df["city"].astype(str).str.strip()
    df["route_id"] = df["route_id"].astype(str).str.strip()

keys = ["city", "route_id"]

# Check for duplicate route records before merging
for name, df in [
    ("Route profiles", profiles),
    ("Service days", service_days),
    ("Distances", distances)
]:
    duplicates = df.duplicated(keys).sum()
    print(name, "- duplicate route records:", duplicates)

    if duplicates > 0:
        df.drop_duplicates(keys, keep="first", inplace=True)

# Prepare hourly departure counts
hourly["departure_clock_hour"] = pd.to_numeric(
    hourly["departure_clock_hour"], errors="coerce"
)

hourly["scheduled_trip_count"] = pd.to_numeric(
    hourly["scheduled_trip_count"], errors="coerce"
)

hourly = hourly.dropna(
    subset=["departure_clock_hour", "scheduled_trip_count"]
)

hourly["departure_clock_hour"] = (
    hourly["departure_clock_hour"].astype(int)
)

hourly = hourly[
    hourly["departure_clock_hour"].between(0, 23)
]

hourly = (
    hourly.groupby(
        ["city", "route_id", "departure_clock_hour"],
        as_index=False
    )["scheduled_trip_count"]
    .sum()
)

# Remember which routes have hourly data
hourly_available = hourly[keys].drop_duplicates().copy()
hourly_available["hourly_departure_data_available"] = True

# Convert hourly data into one row per route
hourly_wide = hourly.pivot(
    index=keys,
    columns="departure_clock_hour",
    values="scheduled_trip_count"
)

# Ensure all 24 clock-hour columns exist
hourly_wide = hourly_wide.reindex(
    columns=range(24), fill_value=0
).fillna(0)

hourly_wide.columns = [
    f"scheduled_departures_hour_{int(hour):02d}"
    for hour in hourly_wide.columns
]

hourly_wide = hourly_wide.reset_index()

# Combine route profiles, service days, hourly counts, and distances
final_df = profiles.merge(
    service_days,
    on=keys,
    how="left",
    suffixes=("", "_service")
)

final_df = final_df.merge(
    hourly_wide,
    on=keys,
    how="left"
)

final_df = final_df.merge(
    hourly_available,
    on=keys,
    how="left"
)

final_df = final_df.merge(
    distances,
    on=keys,
    how="left",
    suffixes=("", "_distance")
)

hour_columns = [
    f"scheduled_departures_hour_{hour:02d}"
    for hour in range(24)
]

# Zero means no departures were recorded in that hour
# for routes with available hourly data.
final_df[hour_columns] = final_df[hour_columns].fillna(0)

final_df["hourly_departure_data_available"] = (
    final_df["hourly_departure_data_available"]
    .fillna(False)
    .astype(bool)
)

# Calculate useful summary features
final_df["total_scheduled_departures_observed"] = (
    final_df[hour_columns].sum(axis=1)
)

final_df.loc[
    ~final_df["hourly_departure_data_available"],
    "total_scheduled_departures_observed"
] = np.nan

final_df["peak_hour_scheduled_departures"] = (
    final_df[hour_columns].max(axis=1)
)

peak_columns = final_df[hour_columns].idxmax(axis=1)

final_df["peak_departure_clock_hour"] = (
    peak_columns.str.extract(r"hour_(\d+)$")[0]
)

final_df.loc[
    ~final_df["hourly_departure_data_available"],
    "peak_departure_clock_hour"
] = np.nan

# Round distance values without filling missing distances
if "estimated_route_length_km" in final_df.columns:
    final_df["estimated_route_length_km"] = (
        pd.to_numeric(
            final_df["estimated_route_length_km"],
            errors="coerce"
        ).round(3)
    )

# Save the final feature-engineered dataset
dataset_path = processed / "module2_feature_engineered_dataset.csv"
profile_path = processed / "module2_route_profiles_final_all_cities.csv"

final_df.to_csv(dataset_path, index=False)
final_df.to_csv(profile_path, index=False)

print("\nFINAL DATASET CREATED SUCCESSFULLY")
print("Total route records:", len(final_df))
print("Cities:", final_df["city"].nunique())
print("Total features:", len(final_df.columns))
print("Missing route distances:",
      final_df["estimated_route_length_km"].isna().sum()
      if "estimated_route_length_km" in final_df.columns
      else "Distance column unavailable")

print("\nDataset saved to:", dataset_path)
print("Route profiles saved to:", profile_path)

display(final_df.head(10))



Route profiles - duplicate route records: 0
Service days - duplicate route records: 0
Distances - duplicate route records: 0


C:\Users\srava\AppData\Local\Temp\ipykernel_18700\1143822653.py:137: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)



FINAL DATASET CREATED SUCCESSFULLY
Total route records: 8472
Cities: 6
Total features: 54
Missing route distances: 1011

Dataset saved to: data\processed\module2_feature_engineered_dataset.csv
Route profiles saved to: data\processed\module2_route_profiles_final_all_cities.csv


,city,route_id,route_name,route_short_name,route_long_name,route_type,distinct_trip_count,distinct_stop_ids_referenced,distinct_known_stops_served,service_ids_used,...,scheduled_departures_hour_20,scheduled_departures_hour_21,scheduled_departures_hour_22,scheduled_departures_hour_23,hourly_departure_data_available,estimated_route_length_km,distance_method,total_scheduled_departures_observed,peak_hour_scheduled_departures,peak_departure_clock_hour
0,Ahmedabad,101sh,101sh,101sh,P.D.P.U. Cross Road ⇆ Asarva Railway Station,3,10,44,44,1.0,...,0.0,0.0,0.0,0.0,True,44.529,Straight-line stop-coordinate estimate,10.0,1.0,06
1,Ahmedabad,103-1,103,103,Bhakti Circle ⇆ Gujarat University,3,22,80,80,1.0,...,0.0,0.0,0.0,0.0,True,22.366,Straight-line stop-coordinate estimate,22.0,2.0,08
2,Ahmedabad,103-2,103,103,Bhakti Circle ⇆ Sarangpur,3,14,52,52,1.0,...,1.0,2.0,2.0,1.0,True,14.922,Straight-line stop-coordinate estimate,14.0,2.0,21
3,Ahmedabad,103-3,103,103,Gujarat University ⇆ Sarangpur,3,6,28,28,1.0,...,1.0,1.0,0.0,0.0,True,8.948,Straight-line stop-coordinate estimate,6.0,2.0,14
4,Ahmedabad,105,105,105,Naroda Industrial Township( MuthiyaGam ) ⇆ Lal...,3,216,40,40,1.0,...,15.0,9.0,7.0,2.0,True,63.727,Straight-line stop-coordinate estimate,216.0,15.0,08
5,Ahmedabad,107,107,107,Sarangpur ⇆ Sarangpur,3,6,91,91,1.0,...,0.0,0.0,0.0,0.0,True,78.545,Straight-line stop-coordinate estimate,6.0,1.0,06
6,Ahmedabad,109-1,109,109,Amba Township Sector-5 (Trimandir) ⇆ G.E.B (Na...,3,6,79,79,1.0,...,0.0,0.0,0.0,0.0,True,74.412,Straight-line stop-coordinate estimate,6.0,2.0,09
7,Ahmedabad,109-2,109,109,Amba Township Sector-5 (Trimandir) ⇆ Naroda,3,3,63,63,1.0,...,0.0,0.0,0.0,0.0,True,66.232,Straight-line stop-coordinate estimate,3.0,1.0,12
8,Ahmedabad,109-3,109,109,Chandkheda Gam ⇆ Amba Township Sector-5 (Trima...,3,1,24,24,1.0,...,0.0,0.0,0.0,0.0,True,10.236,Straight-line stop-coordinate estimate,1.0,1.0,06
9,Ahmedabad,109-4,109,109,Chandkheda Gam ⇆ Naroda,3,4,40,40,1.0,...,0.0,0.0,0.0,0.0,True,55.997,Straight-line stop-coordinate estimate,4.0,1.0,07


In [ ]:
## 10. Create the Feature Dictionary
from pathlib import Path
import pandas as pd

processed = Path("data/processed")

dataset_path = processed / "module2_feature_engineered_dataset.csv"
df = pd.read_csv(dataset_path, low_memory=False)

feature_info = {
    "city": (
        "City where the public transit data was collected",
        "GTFS dataset folder",
        "Assigned from the source city folder",
        "Categorical"
    ),
    "route_id": (
        "Identifier of a transit route",
        "routes.txt",
        "Taken from the GTFS route identifier",
        "Identifier"
    ),
    "route_name": (
        "Display name of the route",
        "routes.txt",
        "Taken from the available route name fields",
        "Categorical"
    ),
    "route_short_name": (
        "Short public-facing route name or number",
        "routes.txt",
        "Taken from route_short_name",
        "Categorical"
    ),
    "route_long_name": (
        "Long descriptive route name",
        "routes.txt",
        "Taken from route_long_name",
        "Categorical"
    ),
    "route_type": (
        "GTFS code describing the transport mode",
        "routes.txt",
        "Taken from route_type; interpreted as a category",
        "Categorical"
    ),
    "distinct_trip_count": (
        "Number of distinct trips associated with the route",
        "trips.txt",
        "Count of unique trip IDs for each route",
        "Integer"
    ),
    "distinct_stop_ids_referenced": (
        "Number of distinct stop IDs referenced in the route's trips",
        "stop_times.txt and trips.txt",
        "Count of unique stop IDs associated with each route",
        "Integer"
    ),
    "distinct_known_stops_served": (
        "Number of distinct referenced stops found in stops.txt",
        "stop_times.txt, trips.txt and stops.txt",
        "Count of unique referenced stop IDs matching the stops table",
        "Integer"
    ),
    "scheduled_operating_days_per_week": (
        "Number of weekdays marked as operating in the available calendar data",
        "calendar.txt",
        "Sum of valid daily service flags when calendar coverage is complete",
        "Integer"
    ),
    "weekday_service": (
        "Indicates whether weekday service is scheduled",
        "calendar.txt",
        "Derived from the Monday-to-Friday calendar flags when reliable",
        "Boolean/Categorical"
    ),
    "weekend_service": (
        "Indicates whether weekend service is scheduled",
        "calendar.txt",
        "Derived from the Saturday and Sunday calendar flags when reliable",
        "Boolean/Categorical"
    ),
    "estimated_route_length_km": (
        "Estimated route length in kilometres; may be unavailable",
        "shapes.txt or stop coordinates in stops.txt",
        "Shape geometry distance when available; otherwise median trip distance estimated from consecutive stop coordinates",
        "Float"
    ),
    "distance_method": (
        "Method used to estimate the route length",
        "shapes.txt or stops.txt",
        "Records whether shape geometry or a straight-line stop-coordinate estimate was used",
        "Categorical"
    ),
    "hourly_departure_data_available": (
        "Indicates whether hourly departure records were available for the route",
        "trips.txt and stop_times.txt",
        "True when the route has valid hourly departure-count records",
        "Boolean"
    ),
    "total_scheduled_departures_observed": (
        "Total recorded first-stop departures across the 24 clock-hour groups",
        "trips.txt and stop_times.txt",
        "Sum of the 24 hourly scheduled departure counts",
        "Numeric"
    ),
    "peak_hour_scheduled_departures": (
        "Largest scheduled departure count recorded in any clock hour",
        "trips.txt and stop_times.txt",
        "Maximum among the 24 hourly departure-count features",
        "Integer"
    ),
    "peak_departure_clock_hour": (
        "Clock hour with the highest recorded departure count, from 0 to 23",
        "trips.txt and stop_times.txt",
        "Clock-hour feature with the maximum scheduled departure count",
        "Categorical/Integer"
    )
}

rows = []

for column in df.columns:
    if column in feature_info:
        meaning, source, calculation, data_type = feature_info[column]

    elif column.startswith("scheduled_departures_hour_"):
        hour = column.replace("scheduled_departures_hour_", "")
        meaning = (
            f"Number of scheduled first-stop departures grouped into "
            f"clock hour {hour}:00"
        )
        source = "trips.txt and stop_times.txt"
        calculation = (
            "Count of distinct trips by route and first-stop departure "
            "clock hour; GTFS hours above 24 are grouped by clock hour"
        )
        data_type = "Integer count"

    elif column in [
        "monday", "tuesday", "wednesday", "thursday",
        "friday", "saturday", "sunday"
    ]:
        meaning = f"Whether scheduled service operates on {column}"
        source = "calendar.txt"
        calculation = (
            "Calendar service flag; retained only when the route's "
            "calendar coverage and flags are considered reliable"
        )
        data_type = "Boolean/Categorical"

    elif "service" in column.lower() or "calendar" in column.lower():
        meaning = "Service calendar coverage or operating-day summary"
        source = "trips.txt and calendar.txt"
        calculation = (
            "Derived from route trips and matching service calendar records; "
            "see the column name for the specific measure"
        )
        data_type = "Numeric/Categorical"

    else:
        meaning = "Review the source column and its role in the dataset"
        source = "See the corresponding GTFS source file"
        calculation = "Refer to the feature engineering code"
        data_type = str(df[column].dtype)

    # IDs and route labels are not quantitative measurements
    if column in [
        "route_id", "route_name", "route_short_name", "route_long_name"
    ]:
        data_type = "Identifier/Categorical"

    rows.append({
        "feature_name": column,
        "meaning": meaning,
        "source": source,
        "calculation": calculation,
        "data_type": data_type
    })

feature_dictionary = pd.DataFrame(rows)

dictionary_path = processed / "module2_feature_dictionary.csv"
feature_dictionary.to_csv(dictionary_path, index=False)

print("FEATURE DICTIONARY CREATED")
print("Total dataset columns documented:", len(feature_dictionary))
print("Saved to:", dictionary_path)

display(feature_dictionary)


FEATURE DICTIONARY CREATED
Total dataset columns documented: 54
Saved to: data\processed\module2_feature_dictionary.csv


,feature_name,meaning,source,calculation,data_type
0,city,City where the public transit data was collected,GTFS dataset folder,Assigned from the source city folder,Categorical
1,route_id,Identifier of a transit route,routes.txt,Taken from the GTFS route identifier,Identifier/Categorical
2,route_name,Display name of the route,routes.txt,Taken from the available route name fields,Identifier/Categorical
3,route_short_name,Short public-facing route name or number,routes.txt,Taken from route_short_name,Identifier/Categorical
4,route_long_name,Long descriptive route name,routes.txt,Taken from route_long_name,Identifier/Categorical
5,route_type,GTFS code describing the transport mode,routes.txt,Taken from route_type; interpreted as a category,Categorical
6,distinct_trip_count,Number of distinct trips associated with the r...,trips.txt,Count of unique trip IDs for each route,Integer
7,distinct_stop_ids_referenced,Number of distinct stop IDs referenced in the ...,stop_times.txt and trips.txt,Count of unique stop IDs associated with each ...,Integer
8,distinct_known_stops_served,Number of distinct referenced stops found in s...,"stop_times.txt, trips.txt and stops.txt",Count of unique referenced stop IDs matching t...,Integer
9,service_ids_used,Service calendar coverage or operating-day sum...,trips.txt and calendar.txt,Derived from route trips and matching service ...,Numeric/Categorical


In [ ]:
## 11. Run Final Data Quality Checks
from pathlib import Path
import pandas as pd
import numpy as np
import re

raw_folder = Path("data/raw")
processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)

results = []


def add_check(city, check, issues, checked, details=""):
    issues = int(issues) if pd.notna(issues) else 0
    checked = int(checked) if pd.notna(checked) else 0

    results.append({
        "city": city,
        "check": check,
        "issue_count": issues,
        "records_checked": checked,
        "issue_percentage": round(
            issues / checked * 100, 2
        ) if checked else np.nan,
        "status": "PASS" if issues == 0 else "REVIEW",
        "details": details
    })


def read_table(path, columns=None):
    if not path.exists():
        return None

    try:
        return pd.read_csv(
            path,
            dtype=str,
            usecols=columns,
            low_memory=False
        )
    except ValueError:
        return pd.read_csv(path, dtype=str, low_memory=False)


def missing_mask(series):
    return series.isna() | series.astype(str).str.strip().eq("")


def invalid_time_mask(series):
    values = series.astype("string").str.strip()
    parts = values.str.extract(
        r"^(\d+):(\d{2}):(\d{2})$"
    )

    hours = pd.to_numeric(parts[0], errors="coerce")
    minutes = pd.to_numeric(parts[1], errors="coerce")
    seconds = pd.to_numeric(parts[2], errors="coerce")

    valid = (
        hours.notna()
        & minutes.between(0, 59)
        & seconds.between(0, 59)
    )

    return ~missing_mask(series) & ~valid


for city_folder in sorted(raw_folder.iterdir()):
    if not city_folder.is_dir():
        continue

    city = city_folder.name
    print("Checking:", city)

    routes = read_table(city_folder / "routes.txt")
    trips = read_table(city_folder / "trips.txt")
    stops = read_table(city_folder / "stops.txt")

    stop_times = read_table(
        city_folder / "stop_times.txt",
        ["trip_id", "stop_id", "stop_sequence",
         "arrival_time", "departure_time"]
    )

    calendar = read_table(city_folder / "calendar.txt")
    shapes = read_table(
        city_folder / "shapes.txt",
        ["shape_id"]
    )

    # Routes: missing and duplicate identifiers
    if routes is not None and "route_id" in routes.columns:
        add_check(
            city, "Missing route_id",
            missing_mask(routes["route_id"]).sum(),
            len(routes)
        )

        valid_routes = routes.loc[
            ~missing_mask(routes["route_id"]), "route_id"
        ]

        add_check(
            city, "Duplicate route_id",
            valid_routes.duplicated().sum(),
            len(valid_routes)
        )

    # Trips: identifiers and route relationships
    if trips is not None and "trip_id" in trips.columns:
        add_check(
            city, "Missing trip_id",
            missing_mask(trips["trip_id"]).sum(),
            len(trips)
        )

        valid_trips = trips.loc[
            ~missing_mask(trips["trip_id"]), "trip_id"
        ]

        add_check(
            city, "Duplicate trip_id",
            valid_trips.duplicated().sum(),
            len(valid_trips)
        )

        if routes is not None and "route_id" in trips.columns:
            known_routes = set(
                routes["route_id"].dropna().astype(str)
            )

            trip_routes = trips["route_id"]
            orphan_routes = (
                ~missing_mask(trip_routes)
                & ~trip_routes.isin(known_routes)
            )

            add_check(
                city, "Trips referencing unknown routes",
                orphan_routes.sum(), len(trips)
            )

            add_check(
                city, "Trips with missing route_id",
                missing_mask(trip_routes).sum(), len(trips)
            )

        if "service_id" in trips.columns:
            add_check(
                city, "Trips with missing service_id",
                missing_mask(trips["service_id"]).sum(),
                len(trips)
            )

            if calendar is not None and "service_id" in calendar.columns:
                known_services = set(
                    calendar["service_id"].dropna().astype(str)
                )

                trip_services = trips["service_id"]
                unmatched = (
                    ~missing_mask(trip_services)
                    & ~trip_services.isin(known_services)
                )

                add_check(
                    city, "Trip service_id absent from calendar.txt",
                    unmatched.sum(), len(trips),
                    "calendar_dates.txt exceptions are not checked here"
                )

        if "shape_id" in trips.columns:
            missing_shapes = missing_mask(trips["shape_id"]).sum()

            if shapes is not None and "shape_id" in shapes.columns:
                known_shapes = set(
                    shapes["shape_id"].dropna().astype(str)
                )

                shape_values = trips["shape_id"]
                unmatched_shapes = (
                    ~missing_mask(shape_values)
                    & ~shape_values.isin(known_shapes)
                )

                add_check(
                    city, "Trip shape_id absent from shapes.txt",
                    unmatched_shapes.sum(), len(trips)
                )

            add_check(
                city, "Trips with missing shape_id",
                missing_shapes, len(trips),
                "Missing shapes are not necessarily errors"
            )

    # Stops: identifiers and coordinates
    if stops is not None:
        if "stop_id" in stops.columns:
            add_check(
                city, "Missing stop_id",
                missing_mask(stops["stop_id"]).sum(),
                len(stops)
            )

            valid_stop_ids = stops.loc[
                ~missing_mask(stops["stop_id"]), "stop_id"
            ]

            add_check(
                city, "Duplicate stop_id",
                valid_stop_ids.duplicated().sum(),
                len(valid_stop_ids)
            )

        if {"stop_lat", "stop_lon"}.issubset(stops.columns):
            lat = pd.to_numeric(stops["stop_lat"], errors="coerce")
            lon = pd.to_numeric(stops["stop_lon"], errors="coerce")

            bad_coordinates = (
                lat.isna() | lon.isna()
                | ~lat.between(-90, 90)
                | ~lon.between(-180, 180)
            )

            add_check(
                city, "Missing or invalid stop coordinates",
                bad_coordinates.sum(), len(stops)
            )

    # Stop times: foreign keys, sequence and times
    if stop_times is not None:
        if trips is not None and "trip_id" in trips.columns:
            known_trip_ids = set(
                trips["trip_id"].dropna().astype(str)
            )

            unknown_trip = (
                ~missing_mask(stop_times["trip_id"])
                & ~stop_times["trip_id"].isin(known_trip_ids)
            )

            add_check(
                city, "Stop times referencing unknown trips",
                unknown_trip.sum(), len(stop_times)
            )

        if stops is not None and "stop_id" in stops.columns:
            known_stop_ids = set(
                stops["stop_id"].dropna().astype(str)
            )

            unknown_stop = (
                ~missing_mask(stop_times["stop_id"])
                & ~stop_times["stop_id"].isin(known_stop_ids)
            )

            add_check(
                city, "Stop times referencing unknown stops",
                unknown_stop.sum(), len(stop_times)
            )

        if {"trip_id", "stop_sequence"}.issubset(stop_times.columns):
            sequence = pd.to_numeric(
                stop_times["stop_sequence"], errors="coerce"
            )

            invalid_sequence = (
                sequence.isna() | (sequence < 0)
            )

            add_check(
                city, "Missing or invalid stop_sequence",
                invalid_sequence.sum(), len(stop_times)
            )

            duplicate_key = (
                ~missing_mask(stop_times["trip_id"])
                & ~missing_mask(stop_times["stop_sequence"])
                & stop_times.duplicated(
                    ["trip_id", "stop_sequence"], keep="first"
                )
            )

            add_check(
                city, "Duplicate trip_id and stop_sequence",
                duplicate_key.sum(), len(stop_times)
            )

        for time_column in ["arrival_time", "departure_time"]:
            if time_column in stop_times.columns:
                missing_time = missing_mask(
                    stop_times[time_column]
                ).sum()

                invalid_time = invalid_time_mask(
                    stop_times[time_column]
                ).sum()

                add_check(
                    city, f"Missing {time_column}",
                    missing_time, len(stop_times)
                )

                add_check(
                    city, f"Invalid {time_column} format or range",
                    invalid_time, len(stop_times),
                    "GTFS hours above 24 are permitted"
                )

# Check the final route-level dataset
final_path = processed / "module2_feature_engineered_dataset.csv"

if final_path.exists():
    final_data = pd.read_csv(
        final_path, dtype={"city": str, "route_id": str},
        low_memory=False
    )

    if {"city", "route_id"}.issubset(final_data.columns):
        add_check(
            "ALL CITIES", "Duplicate city and route_id in final dataset",
            final_data.duplicated(
                ["city", "route_id"]
            ).sum(),
            len(final_data)
        )

        add_check(
            "ALL CITIES", "Missing city or route_id in final dataset",
            (
                missing_mask(final_data["city"])
                | missing_mask(final_data["route_id"])
            ).sum(),
            len(final_data)
        )

quality_checks = pd.DataFrame(results)

report_path = processed / "module2_final_data_quality_checks.csv"
quality_checks.to_csv(report_path, index=False)

print("\nFINAL DATA QUALITY CHECKS COMPLETED")
print("Checks performed:", len(quality_checks))
print("Issues requiring review:",
      (quality_checks["status"] == "REVIEW").sum())
print("Report saved to:", report_path)

display(
    quality_checks[
        quality_checks["status"] == "REVIEW"
    ].head(30)
)


Checking: Ahmedabad
Checking: Chennai
Checking: Hyderabad
Checking: Mumbai
Checking: Pune
Checking: Rajkot

FINAL DATA QUALITY CHECKS COMPLETED
Checks performed: 122
Issues requiring review: 2
Report saved to: data\processed\module2_final_data_quality_checks.csv


,city,check,issue_count,records_checked,issue_percentage,status,details
9,Ahmedabad,Trips with missing shape_id,14323,14323,100.00,REVIEW,Missing shapes are not necessarily errors
30,Chennai,Trips with missing shape_id,47810,48435,98.71,REVIEW,Missing shapes are not necessarily errors


In [ ]:
## 12. Document Assumptions and Limitations
from pathlib import Path

processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)

document = """
# RouteRanker — Module 2
## Assumptions and Limitations

### 1. Assumptions

1. The GTFS files supplied for each city represent the available
   public transit schedule information for that dataset.

2. route_id identifies a route within its city, and trip_id
   identifies a scheduled trip within its source dataset.

3. Distinct trip counts represent the number of unique scheduled
   trips associated with a route, not passenger numbers.

4. Stops served are counted using distinct stop IDs referenced
   by the route's trips in stop_times.txt.

5. Scheduled departure activity is measured using the first
   recorded stop of each trip, ordered by stop_sequence.

6. Clock-hour analysis groups departure times into hours 0–23.
   GTFS times above 24 hours are mapped to their corresponding
   clock hour for hourly activity analysis.

7. Operating-day features are calculated from calendar.txt.
   They are considered reliable only where the required service
   calendar records and daily flags are available and valid.

8. Route distance uses GTFS shape geometry when a usable shape
   can be linked to trips. Otherwise, the implementation may
   estimate distance from consecutive stop coordinates.

9. Missing optional GTFS fields are retained as missing when
   they cannot be derived reliably. Missing distances are not
   automatically replaced with zero.

10. Raw GTFS source files are kept separate from processed
    feature-engineered outputs.

### 2. Limitations

1. Some city datasets do not contain calendar_dates.txt.
   Service exceptions and special-date changes are therefore
   not fully represented in the operating-day analysis.

2. Some datasets lack shapes.txt or have trips with missing
   or unmatched shape_id values. Reliable shape-based distance
   estimates may not be available for every route.

3. Stop-coordinate distance estimates use straight-line
   distances between consecutive stops. They do not represent
   actual road distance and may underestimate the true journey.

4. A route's estimated distance may vary across trips. The
   route-level value is a summary of available trip estimates,
   not a guarantee that every trip follows the same path.

5. Hourly departure counts describe scheduled trips recorded
   in the supplied GTFS data. They do not establish actual
   service operation, delays, cancellations, or real-time traffic.

6. Trip counts are not the same as trip frequency or headway.
   Headway requires the time differences between consecutive
   scheduled departures and has not been calculated here.

7. GTFS schedule data does not provide passenger counts or
   vehicle occupancy. It cannot independently establish
   passenger demand or overcrowding.

8. Missing values, inconsistent identifiers, and duplicate
   records may affect individual calculations. The data quality
   reports should be reviewed before using the dataset for
   further modelling.

9. Calendar service analysis uses calendar.txt. Because
   calendar_dates.txt exceptions are not incorporated into
   the current operating-day calculation, special service
   dates may be missed.

10. Route-ranking and shortest-path recommendations are not
    established by this feature-engineering dataset alone.
    A separate routing and ranking method is required.

### 3. Interpretation Notes

- Route IDs are identifiers, not numerical measurements.
- A missing distance means that a usable estimate was unavailable.
- An hourly count of zero means no departure was recorded in
  that hour when hourly data is available.
- A missing operating-day value should not be interpreted as
  zero operating days.
- Results depend on the completeness and consistency of the
  source GTFS data.

### 4. Output Files

The Module 2 processing workflow produces route profiles,
time-feature files, hourly departure counts, operating-day
summaries, distance estimates, a combined feature-engineered
dataset, a feature dictionary, and data quality reports.

These outputs should be checked before they are used in
subsequent RouteRanker modules.
"""

output_file = processed / "module2_assumptions_and_limitations.md"
output_file.write_text(document.strip() + "\n", encoding="utf-8")

print("Assumptions and Limitations document created!")
print("Saved to:", output_file)
print("\nDocument preview:\n")
print(document)


Assumptions and Limitations document created!
Saved to: data\processed\module2_assumptions_and_limitations.md

Document preview:


# RouteRanker — Module 2
## Assumptions and Limitations

### 1. Assumptions

1. The GTFS files supplied for each city represent the available
   public transit schedule information for that dataset.

2. route_id identifies a route within its city, and trip_id
   identifies a scheduled trip within its source dataset.

3. Distinct trip counts represent the number of unique scheduled
   trips associated with a route, not passenger numbers.

4. Stops served are counted using distinct stop IDs referenced
   by the route's trips in stop_times.txt.

5. Scheduled departure activity is measured using the first
   recorded stop of each trip, ordered by stop_sequence.

6. Clock-hour analysis groups departure times into hours 0–23.
   GTFS times above 24 hours are mapped to their corresponding
   clock hour for hourly activity analysis.

7. Operating-day features are c

In [ ]:
## 13. Verify Module 2 Deliverables and Final Dataset
from pathlib import Path
import pandas as pd

raw_folder = Path("data/raw")
processed = Path("data/processed")

# Required Module 2 deliverables
required_files = [
    "module2_feature_engineered_dataset.csv",
    "module2_route_profiles_final_all_cities.csv",
    "module2_feature_dictionary.csv",
    "module2_final_data_quality_checks.csv",
    "module2_data_quality_report.csv",
    "module2_missing_values.csv",
    "module2_route_profiles_basic_all_cities.csv",
    "module2_hourly_trip_counts_all_cities.csv",
    "module2_route_service_days_all_cities.csv",
    "module2_route_distance_estimates.csv",
    "module2_assumptions_and_limitations.md"
]

verification = []

for filename in required_files:
    path = processed / filename
    verification.append({
        "item": filename,
        "status": "PASS" if path.exists() else "MISSING",
        "details": (
            f"{path.stat().st_size:,} bytes"
            if path.exists() else "File not found"
        )
    })

# Check time-feature files for each raw city folder
city_folders = sorted(
    p for p in raw_folder.iterdir() if p.is_dir()
)

for city_folder in city_folders:
    time_file = (
        processed
        / f"{city_folder.name.lower()}_stop_times_time_features.csv"
    )

    verification.append({
        "item": f"{city_folder.name} time-feature file",
        "status": "PASS" if time_file.exists() else "MISSING",
        "details": (
            f"{time_file.stat().st_size:,} bytes"
            if time_file.exists() else "File not found"
        )
    })

# Check final dataset and feature dictionary
dataset_path = processed / "module2_feature_engineered_dataset.csv"
dictionary_path = processed / "module2_feature_dictionary.csv"

if dataset_path.exists() and dictionary_path.exists():
    df = pd.read_csv(dataset_path, low_memory=False)
    dictionary = pd.read_csv(dictionary_path, low_memory=False)

    dataset_columns = set(df.columns)
    dictionary_columns = set(dictionary["feature_name"].dropna())

    missing_descriptions = sorted(
        dataset_columns - dictionary_columns
    )
    extra_descriptions = sorted(
        dictionary_columns - dataset_columns
    )

    duplicate_routes = df.duplicated(
        ["city", "route_id"]
    ).sum()

    print("FINAL DATASET SUMMARY")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Cities represented:", df["city"].nunique())
    print("Duplicate city-route pairs:", duplicate_routes)
    print("Dataset columns without dictionary entries:",
          missing_descriptions)
    print("Dictionary entries not in dataset:",
          extra_descriptions)

    print("\nROUTES BY CITY")
    display(
        df.groupby("city")
        .size()
        .reset_index(name="route_records")
    )

    verification.append({
        "item": "Feature dictionary covers dataset columns",
        "status": (
            "PASS" if not missing_descriptions
            and not extra_descriptions else "REVIEW"
        ),
        "details": (
            f"Undocumented: {len(missing_descriptions)}; "
            f"extra entries: {len(extra_descriptions)}"
        )
    })

    verification.append({
        "item": "Unique city-route pairs",
        "status": "PASS" if duplicate_routes == 0 else "REVIEW",
        "details": f"{duplicate_routes} duplicate pairs"
    })

else:
    print("Cannot compare dataset and dictionary: one or both files are missing.")

verification_df = pd.DataFrame(verification)

verification_path = processed / "module2_final_verification_report.csv"
verification_df.to_csv(verification_path, index=False)

print("\nFINAL VERIFICATION REPORT")
display(verification_df)

print("\nPASS:", (verification_df["status"] == "PASS").sum())
print("MISSING:", (verification_df["status"] == "MISSING").sum())
print("REVIEW:", (verification_df["status"] == "REVIEW").sum())
print("Saved to:", verification_path)


FINAL DATASET SUMMARY
Rows: 8472
Columns: 54
Cities represented: 6
Duplicate city-route pairs: 0
Dataset columns without dictionary entries: []
Dictionary entries not in dataset: []

ROUTES BY CITY


,city,route_records
0,Ahmedabad,639
1,Chennai,5402
2,Hyderabad,1031
3,Mumbai,935
4,Pune,315
5,Rajkot,150



FINAL VERIFICATION REPORT


,item,status,details
0,module2_feature_engineered_dataset.csv,PASS,"2,120,775 bytes"
1,module2_route_profiles_final_all_cities.csv,PASS,"2,120,775 bytes"
2,module2_feature_dictionary.csv,PASS,"11,693 bytes"
3,module2_final_data_quality_checks.csv,PASS,"7,693 bytes"
4,module2_data_quality_report.csv,PASS,"1,160 bytes"
5,module2_missing_values.csv,PASS,638 bytes
6,module2_route_profiles_basic_all_cities.csv,PASS,"575,474 bytes"
7,module2_hourly_trip_counts_all_cities.csv,PASS,"1,093,361 bytes"
8,module2_route_service_days_all_cities.csv,PASS,"311,071 bytes"
9,module2_route_distance_estimates.csv,PASS,"553,770 bytes"



PASS: 19
MISSING: 0
REVIEW: 0
Saved to: data\processed\module2_final_verification_report.csv
